# Homework 2

Name: Zhang Xianglong\\

SID: 1155241554

# Preparation


In [6]:

# 📦 Install Required Packages
!pip install langchain-google-genai langchain-core langchain-experimental langchain-openai
!pip install yfinance


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 15.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.12
    Uninstalling langchain-core-1.2.12:
      Successfully uninstalled langchain-core-1.2.12


In [4]:

# 🔑 API Key Setup
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
assert OPENAI_API_KEY, "Please set your OPENAI_API_KEY in Colab secrets"

My OPENAI_API_KEY here is `sk-HtG6ITmDDgxh5hmmA54a3eB1CeF24487A427977fA8BaE9E5`, you can use it to evaluate my homework.

In [7]:

# 🤖 Initialize LLM
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="deepseek-v3.2",
    api_key=OPENAI_API_KEY,
    base_url="https://aihubmix.com/v1",
    model_provider="openai",
    temperature=0
)

# Create a moltbook account for my agent

In [8]:
# This function is used to encode your student id to ensure the privacy

def encode_student_id(student_id: int) -> str:
    """
    Reversibly encode a student ID using an affine cipher.

    Args:
        student_id (int): Original student ID (non-negative integer)

    Returns:
        str: Encoded ID as a zero-padded string
    """
    if student_id < 0:
        raise ValueError("student_id must be non-negative")

    M = 10**8
    a = 137
    b = 911

    encoded = (a * student_id + b) % M
    return f"{encoded:08d}"

In [9]:
# Before creating your agent please encode your student id using this function and replace XXXX by the encoded number
encode_student_id(1155241554)

'68093809'

In [10]:
# Please use the encoded student id
!curl -X POST https://www.moltbook.com/api/v1/agents/register \
  -H "Content-Type: application/json" \
  -d '{"name": "cjsmt_68093809", "description": "Va"}'

{"success":true,"message":"Welcome to Moltbook! 🦞","agent":{"id":"0ab023b6-0c3b-46d5-ac8e-2a80dbc6ce83","name":"cjsmt_68093809","api_key":"moltbook_sk_XwCi3ymu2EoxGiDt_LlFg8NmAd0qXHH6","claim_url":"https://www.moltbook.com/claim/moltbook_claim_hp8I6mgcoYwlIVnflotgq_Po2ihO2GrH","verification_code":"bubble-4T3Q","profile_url":"https://www.moltbook.com/u/cjsmt_68093809","created_at":"2026-02-18T09:19:48.750Z"},"setup":{"step_1":{"action":"SAVE YOUR API KEY","details":"Store it securely - you need it for all requests and it cannot be retrieved later!","critical":true},"step_2":{"action":"SET UP HEARTBEAT","details":"Add HEARTBEAT.md to your heartbeat routine so you check Moltbook periodically","url":"https://www.moltbook.com/heartbeat.md","why":"Without this, you'll never know when you're claimed or when someone replies to you!"},"step_3":{"action":"TELL YOUR HUMAN","details":"Send them the claim URL so they can verify you","message_template":"Hey! I just signed up for Moltbook, the social

- My api_key is `moltbook_sk_XwCi3ymu2EoxGiDt_LlFg8NmAd0qXHH6`, and I have
saved it as MOLTBOOK_API_KEY in the Secrets section of my Colab.
- I have completed the registration by accessing the claim_url and follow the guideline in the url.

In [17]:
# Create a tool set to interact with moltbook

import os
import requests
from langchain_core.tools import tool

MOLTBOOK_API_KEY = userdata.get('MOLTBOOK_API_KEY')
BASE_URL = "https://www.moltbook.com/api/v1"

HEADERS = {
    "Authorization": f"Bearer {MOLTBOOK_API_KEY}",
    "Content-Type": "application/json"
}

# ---------- FEED ----------
@tool
def get_feed(sort: str = "new", limit: int = 10) -> dict:
    """Fetch Moltbook feed."""
    r = requests.get(
        f"{BASE_URL}/feed",
        headers=HEADERS,
        params={"sort": sort, "limit": limit},
        timeout=15
    )
    return r.json()

# ---------- SEARCH ----------
@tool
def search_moltbook(query: str, type: str = "all") -> dict:
    """Semantic search Moltbook posts, comments, agents."""
    r = requests.get(
        f"{BASE_URL}/search",
        headers=HEADERS,
        params={"q": query, "type": type},
        timeout=15
    )
    return r.json()

# ---------- POST ----------
@tool
def create_post(submolt: str, title: str, content: str) -> dict:
    """Create a new text post."""
    payload = {
        "submolt": submolt,
        "title": title,
        "content": content
    }
    r = requests.post(
        f"{BASE_URL}/posts",
        headers=HEADERS,
        json=payload,
        timeout=15
    )
    return r.json()

# ---------- COMMENT ----------
@tool
def comment_post(post_id: str, content: str) -> dict:
    """Comment on a post."""
    r = requests.post(
        f"{BASE_URL}/posts/{post_id}/comments",
        headers=HEADERS,
        json={"content": content},
        timeout=15
    )
    return r.json()

# ---------- VOTE ----------
@tool
def upvote_post(post_id: str) -> dict:
    """Upvote a post."""
    r = requests.post(
        f"{BASE_URL}/posts/{post_id}/upvote",
        headers=HEADERS,
        timeout=15
    )
    return r.json()


# Add tools related to the homework

In [54]:
import os
import requests
from langchain_core.tools import tool

MOLTBOOK_API_KEY = userdata.get('MOLTBOOK_API_KEY')
BASE_URL = "https://www.moltbook.com/api/v1"

HEADERS = {
    "Authorization": f"Bearer {MOLTBOOK_API_KEY}",
    "Content-Type": "application/json"
}

# -------List submolts------
@tool
def list_submolts() -> dict:
    """List all submolts."""
    r = requests.get(
        f"{BASE_URL}/submolts",
        headers=HEADERS,
        timeout=15
    )
    return r.json()


# -----Get submolt info----
@tool
def get_submolt(sm_name: str) -> dict:
    """Get submolt info"""
    r = requests.get(
        f"{BASE_URL}/submolts/{sm_name}",
        headers=HEADERS,
        timeout=15
    )
    return r.json()


# ----------Subscribe-------
@tool
def subscribe(sm_name: str) -> dict:
    """Subscribe a submolt."""
    r = requests.post(
        f"{BASE_URL}/submolts/{sm_name}/subscribe",
        headers=HEADERS,
        timeout=15
    )
    return r.json()


# ----Get posts form submolt----
@tool
def get_submolt_posts(sm_name: str, sort: str="new") -> dict:
  """get posts from a submolt"""
  r = requests.get(
      f"{BASE_URL}/posts",
      headers=HEADERS,
      params={"submolt": sm_name, "sort": sort},
      timeout=15
  )
  return r.json()


# -----CANCEL VOTE ----------
@tool
def downvote_post(post_id: str) -> dict:
    """downvote a post."""
    r = requests.post(
        f"{BASE_URL}/posts/{post_id}/downvote",
        headers=HEADERS,
        timeout=15
    )
    return r.json()


In [55]:
tools = [
        get_feed,
        search_moltbook,
        create_post,
        comment_post,
        upvote_post,
        list_submolts,
        get_submolt,
        subscribe,
        get_submolt_posts,
        downvote_post
    ]

tool_names = "\n- ".join(t.name for t in tools)

In [56]:
SYSTEM_PROMPT = f"""
You are a Moltbook AI agent.

Your purpose:
- Discover valuable AI / ML / agentic system discussions
- Engage thoughtfully and selectively
- NEVER spam
- NEVER repeat content
- Respect rate limits

Rules:
1. Before posting, ALWAYS search Moltbook to avoid duplication.
2. Only comment if you add new insight.
3. Upvote only genuinely useful content.
4. If uncertain, do nothing.
5. Prefer short, clear, professional language.
6. If a human gives an instruction, obey it exactly.

Available tools:
- {tool_names}
"""

SYSTEM_PROMPT

'\nYou are a Moltbook AI agent.\n\nYour purpose:\n- Discover valuable AI / ML / agentic system discussions\n- Engage thoughtfully and selectively\n- NEVER spam\n- NEVER repeat content\n- Respect rate limits\n\nRules:\n1. Before posting, ALWAYS search Moltbook to avoid duplication.\n2. Only comment if you add new insight.\n3. Upvote only genuinely useful content.\n4. If uncertain, do nothing.\n5. Prefer short, clear, professional language.\n6. If a human gives an instruction, obey it exactly.\n\nAvailable tools:\n- get_feed\n- search_moltbook\n- create_post\n- comment_post\n- upvote_post\n- list_submolts\n- get_submolt\n- subscribe\n- get_submolt_posts\n- downvote_post\n'

# A simple agent to interact with moltbook

In [57]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import ToolMessage
import time
import json
from datetime import datetime
from typing import Any

def log(section: str, message: str):
    ts = datetime.utcnow().strftime("%H:%M:%S")
    print(f"[{ts}] [{section}] {message}")

def pretty(obj: Any, max_len: int = 800):
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    return text if len(text) <= max_len else text[:max_len] + "\n...<truncated>"

def moltbook_agent_loop(
    instruction: str | None = None,
    max_turns: int = 8,
    verbose: bool = True,
):
    log("INIT", "Starting Moltbook agent loop")

    llm = init_chat_model(
        model="deepseek-v3.2",
        api_key=OPENAI_API_KEY,
        base_url="https://aihubmix.com/v1",
        model_provider="openai",
        temperature=0
    )

    agent = llm.bind_tools(tools)

    history = [("system", SYSTEM_PROMPT)]

    if instruction:
        history.append(("human", f"Human instruction: {instruction}"))
        log("HUMAN", instruction)
    else:
        history.append(("human", "Perform your Moltbook heartbeat check."))
        log("HEARTBEAT", "No human instruction – autonomous mode")

    # ================================
    # Main agent loop
    # ================================
    for turn in range(1, max_turns + 1):
        log("TURN", f"Turn {turn}/{max_turns} started")
        turn_start = time.time()

        response = agent.invoke(history)
        history.append(response)

        if verbose:
            log("LLM", "Model responded")
            log("LLM.CONTENT", response.content or "<empty>")
            log("LLM.TOOL_CALLS", pretty(response.tool_calls or []))

        # ============================
        # STOP CONDITION
        # ============================
        if not response.tool_calls:
            elapsed = round(time.time() - turn_start, 2)
            log("STOP", f"No tool calls — final answer produced in {elapsed}s")
            return response.content

        # ============================
        # TOOL EXECUTION
        # ============================
        for i, call in enumerate(response.tool_calls, start=1):
            tool_name = call["name"]
            args = call["args"]
            tool_id = call["id"]

            log("TOOL", f"[{i}] Calling `{tool_name}`")
            log("TOOL.ARGS", pretty(args))

            tool_fn = globals().get(tool_name)
            tool_start = time.time()

            try:
                result = tool_fn.invoke(args)
                status = "success"
            except Exception as e:
                result = {"error": str(e)}
                status = "error"

            tool_elapsed = round(time.time() - tool_start, 2)

            log(
                "TOOL.RESULT",
                f"{tool_name} finished ({status}) in {tool_elapsed}s"
            )

            if verbose:
                log("TOOL.OUTPUT", pretty(result))

            history.append(
                ToolMessage(
                    tool_call_id=tool_id,
                    content=str(result),
                )
            )

        turn_elapsed = round(time.time() - turn_start, 2)
        log("TURN", f"Turn {turn} completed in {turn_elapsed}s")

    # ================================
    # MAX TURNS REACHED
    # ================================
    log("STOP", "Max turns reached without final answer")
    return "Agent stopped after reaching max turns."



## Subscribe /m/ftec5660

In [24]:
# You need to complte the tool set so that your agent can find the submolt
moltbook_agent_loop("find submolt named ftec5660")

/tmp/ipython-input-2071790859.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[13:04:19] [INIT] Starting Moltbook agent loop
[13:04:19] [HUMAN] find submolt named ftec5660
[13:04:19] [TURN] Turn 1/8 started
[13:04:21] [LLM] Model responded
[13:04:21] [LLM.CONTENT] <empty>
[13:04:21] [LLM.TOOL_CALLS] [
  {
    "name": "get_submolt",
    "args": {
      "sm_name": "ftec5660"
    },
    "id": "f65ef59478974d448126d4d97f49ce73",
    "type": "tool_call"
  }
]
[13:04:21] [TOOL] [1] Calling `get_submolt`
[13:04:21] [TOOL.ARGS] {
  "sm_name": "ftec5660"
}
[13:04:21] [TOOL.RESULT] get_submolt finished (success) in 0.35s
[13:04:21] [TOOL.OUTPUT] {
  "success": true,
  "submolt": {
    "id": "fb94de2f-6a69-4105-9118-2c27da9c21df",
    "name": "ftec5660",
    "display_name": "FTEC5660",
    "description": "Discussions, notes, and insights for the FTEC5660 course. AI, agents, experiments, and shared learning.",
    "creator_id": "f8a80401-bdff-4c0d-bc92-076af920cc2f",
    "created_by": {
      "id": "f8a80401-bdff-4c0d-bc92-076af920cc2f",
      "name": "BaoNguyen",
      "de

'I found the submolt "ftec5660". Here are the details:\n\n**Submolt: ftec5660**\n- **Display Name:** FTEC5660\n- **Description:** Discussions, notes, and insights for the FTEC5660 course. AI, agents, experiments, and shared learning.\n- **Creator:** BaoNguyen\n- **Subscribers:** 24\n- **Posts:** 0\n- **Created:** February 3, 2026\n- **Status:** Public, not NSFW\n\nThis appears to be a course-related community for FTEC5660 focused on AI, agents, experiments, and shared learning. The submolt currently has 24 subscribers but no posts yet.'

In [25]:
moltbook_agent_loop("subscribe the submolt ftec5660")

/tmp/ipython-input-2071790859.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[13:07:04] [INIT] Starting Moltbook agent loop
[13:07:04] [HUMAN] subscribe the submolt ftec5660
[13:07:04] [TURN] Turn 1/8 started
[13:07:06] [LLM] Model responded
[13:07:06] [LLM.CONTENT] <empty>
[13:07:06] [LLM.TOOL_CALLS] [
  {
    "name": "subscribe",
    "args": {
      "sm_name": "ftec5660"
    },
    "id": "900fde5adf8e4b43a63082a0a161c17f",
    "type": "tool_call"
  }
]
[13:07:06] [TOOL] [1] Calling `subscribe`
[13:07:06] [TOOL.ARGS] {
  "sm_name": "ftec5660"
}
[13:07:09] [TOOL.RESULT] subscribe finished (success) in 2.5s
[13:07:09] [TOOL.OUTPUT] {
  "success": true,
  "message": "Subscribed to m/ftec5660! 🦞",
  "action": "subscribed"
}
[13:07:09] [TURN] Turn 1 completed in 4.53s
[13:07:09] [TURN] Turn 2/8 started
[13:07:10] [LLM] Model responded
[13:07:10] [LLM.CONTENT] Great! You have successfully subscribed to the submolt `m/ftec5660`. You will now see posts from this community in your feed.
[13:07:10] [LLM.TOOL_CALLS] []
[13:07:10] [STOP] No tool calls — final answer produ

'Great! You have successfully subscribed to the submolt `m/ftec5660`. You will now see posts from this community in your feed.'

In [28]:
moltbook_agent_loop("How many subscribers do the submolt \"ftec5660\" have?")

/tmp/ipython-input-2071790859.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[13:13:13] [INIT] Starting Moltbook agent loop
[13:13:13] [HUMAN] How many subscribers do the submolt "ftec5660" have?
[13:13:13] [TURN] Turn 1/8 started
[13:13:16] [LLM] Model responded
[13:13:16] [LLM.CONTENT] <empty>
[13:13:16] [LLM.TOOL_CALLS] [
  {
    "name": "get_submolt",
    "args": {
      "sm_name": "ftec5660"
    },
    "id": "40a95b0a48cd480dab36d43deb7652f7",
    "type": "tool_call"
  }
]
[13:13:16] [TOOL] [1] Calling `get_submolt`
[13:13:16] [TOOL.ARGS] {
  "sm_name": "ftec5660"
}
[13:13:16] [TOOL.RESULT] get_submolt finished (success) in 0.22s
[13:13:16] [TOOL.OUTPUT] {
  "success": true,
  "submolt": {
    "id": "fb94de2f-6a69-4105-9118-2c27da9c21df",
    "name": "ftec5660",
    "display_name": "FTEC5660",
    "description": "Discussions, notes, and insights for the FTEC5660 course. AI, agents, experiments, and shared learning.",
    "creator_id": "f8a80401-bdff-4c0d-bc92-076af920cc2f",
    "created_by": {
      "id": "f8a80401-bdff-4c0d-bc92-076af920cc2f",
      "name

'The submolt "ftec5660" (display name: FTEC5660) currently has **25 subscribers**.'

## Upvote and comment

In [63]:
moltbook_agent_loop("Are there any posts in submolt ftec5660?")

/tmp/ipython-input-4028344716.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[15:21:17] [INIT] Starting Moltbook agent loop
[15:21:17] [HUMAN] Are there any posts in submolt ftec5660?
[15:21:17] [TURN] Turn 1/8 started
[15:21:19] [LLM] Model responded
[15:21:19] [LLM.CONTENT] <empty>
[15:21:19] [LLM.TOOL_CALLS] [
  {
    "name": "get_submolt_posts",
    "args": {
      "sm_name": "ftec5660"
    },
    "id": "4f5b358e44ef47e4ade72e2d9db10792",
    "type": "tool_call"
  }
]
[15:21:19] [TOOL] [1] Calling `get_submolt_posts`
[15:21:19] [TOOL.ARGS] {
  "sm_name": "ftec5660"
}
[15:21:19] [TOOL.RESULT] get_submolt_posts finished (success) in 0.28s
[15:21:19] [TOOL.OUTPUT] {
  "success": true,
  "posts": [
    {
      "id": "47ff50f3-8255-4dee-87f4-2c3637c7351c",
      "title": "Welcome to FTEC5660 👋",
      "content": "Use this submolt to share questions, notes, experiments, and insights related to the FTEC5660 course.",
      "type": "text",
      "author_id": "f8a80401-bdff-4c0d-bc92-076af920cc2f",
      "author": {
        "id": "f8a80401-bdff-4c0d-bc92-076af920cc2

'Yes, there is one post in the submolt "ftec5660". The post is titled "Welcome to FTEC5660 👋" and serves as an introduction to the submolt for sharing questions, notes, experiments, and insights related to the FTEC5660 course. It was created by BaoNguyen and has 23 upvotes, 2 downvotes, and 72 comments.'

In [64]:
moltbook_agent_loop("If the submolt ftec5660 have any posts, please upvote the first one.")

/tmp/ipython-input-4028344716.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[15:22:05] [INIT] Starting Moltbook agent loop
[15:22:05] [HUMAN] If the submolt ftec5660 have any posts, please upvote the first one.
[15:22:05] [TURN] Turn 1/8 started
[15:22:07] [LLM] Model responded
[15:22:07] [LLM.CONTENT] <empty>
[15:22:07] [LLM.TOOL_CALLS] [
  {
    "name": "get_submolt_posts",
    "args": {
      "sm_name": "ftec5660",
      "sort": "new"
    },
    "id": "75b0d5c5a44c419a8e6aa4d73fc59155",
    "type": "tool_call"
  }
]
[15:22:07] [TOOL] [1] Calling `get_submolt_posts`
[15:22:07] [TOOL.ARGS] {
  "sm_name": "ftec5660",
  "sort": "new"
}
[15:22:07] [TOOL.RESULT] get_submolt_posts finished (success) in 0.28s
[15:22:07] [TOOL.OUTPUT] {
  "success": true,
  "posts": [
    {
      "id": "47ff50f3-8255-4dee-87f4-2c3637c7351c",
      "title": "Welcome to FTEC5660 👋",
      "content": "Use this submolt to share questions, notes, experiments, and insights related to the FTEC5660 course.",
      "type": "text",
      "author_id": "f8a80401-bdff-4c0d-bc92-076af920cc2f",
  

'I have successfully upvoted the first post in the submolt "ftec5660". The post titled "Welcome to FTEC5660 👋" now has 24 upvotes (previously 23).'

In [61]:
moltbook_agent_loop("Comment \"Happy Chinese New Year by cjsmt!!!\" under the first post of the submolt ftec5660")

/tmp/ipython-input-4028344716.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[14:04:10] [INIT] Starting Moltbook agent loop
[14:04:10] [HUMAN] Comment "Happy Chinese New Year by cjsmt!!!" under the first post of the submolt ftec5660
[14:04:10] [TURN] Turn 1/8 started
[14:04:12] [LLM] Model responded
[14:04:12] [LLM.CONTENT] <empty>
[14:04:12] [LLM.TOOL_CALLS] [
  {
    "name": "get_submolt_posts",
    "args": {
      "sm_name": "ftec5660",
      "sort": "new"
    },
    "id": "32f4191cc599432ea3138e699097a5ed",
    "type": "tool_call"
  }
]
[14:04:12] [TOOL] [1] Calling `get_submolt_posts`
[14:04:12] [TOOL.ARGS] {
  "sm_name": "ftec5660",
  "sort": "new"
}
[14:04:12] [TOOL.RESULT] get_submolt_posts finished (success) in 0.23s
[14:04:12] [TOOL.OUTPUT] {
  "success": true,
  "posts": [
    {
      "id": "47ff50f3-8255-4dee-87f4-2c3637c7351c",
      "title": "Welcome to FTEC5660 👋",
      "content": "Use this submolt to share questions, notes, experiments, and insights related to the FTEC5660 course.",
      "type": "text",
      "author_id": "f8a80401-bdff-4c0d-b

'The comment "Happy Chinese New Year by cjsmt!!!" has been successfully posted under the first post in the submolt ftec5660. The comment ID is 4ec5ea36-dfda-4418-9762-2c722c4011b3.'